# Whisper → ONNX → TensorRT (whisper2)

`whisper.ipynb`를 정리·수정한 버전. 위에서 아래로 한 번에 돌아가도록 구성.

원본 대비 수정 사항
- **TRT 인코더 입력 버그**: CPU 텐서를 넣으면 `WhisperTRTEncoder.forward`가 mel 특징을 `.int()`로 캐스팅 → 인코더 출력이 틀어짐 ("Encoder output이 다름!!"의 원인). `Whisper/trt.py`에서 `.float()`로 수정.
- **TRT e2e 결과 불일치**: 원본은 TRT 디코더에 *한국어 뉴스*의 인코더 출력(`encoder_outputs`)을 넘기고 librispeech 정답과 비교해서 `False` / `" you"`가 나왔음. 같은 오디오의 TRT 인코더 출력을 쓰도록 수정.
- `cuda:1`/`cuda` 혼용 제거: `CUDA_VISIBLE_DEVICES`로 GPU 하나만 보이게 하고 전부 `cuda`로 통일.
- NNDF가 `archived_models/`로 옮겨져서 import 경로 추가.
- ONNX는 FP32로 한 번만 export하고 FP32/FP16 엔진 둘 다 같은 ONNX에서 빌드.

v2 (opset 17 + KV cache, `Whisper/trt_kv.py`)
- **opset 17 export**: LayerNorm이 ONNX `LayerNormalization` → TRT `INormalizationLayer`로 들어가서 FP16 엔진에서도 정규화는 FP32로 계산됨. large 모델 인코더의 FP16 overflow(LayerNorm 입력 최대 ~480, 제곱 ~23만 > FP16 최대 65504) 때문에 인코더를 통째로 FP32로 돌리던 우회가 필요 없어짐.
- **KV cache**: 인코더 엔진이 디코더 32층의 cross-attention K/V까지 한 번에 계산하고, 디코더 엔진은 self-attention K/V cache를 입출력으로 받아 새 토큰 1개만 계산. (기존 NNDF KV 경로는 BART/T5 코드가 섞여 동작하지 않아서 Whisper 전용으로 새로 작성)

환경: conda `trt` (python 3.8, torch 2.0.1+cu117, transformers 4.24.0, tensorrt 8.6.1, polygraphy 0.48.1, onnx 1.14.1)

In [1]:
import os, sys

# 사용할 GPU 하나만 노출 (torch / tensorrt import 전에 설정해야 함)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("CUDA_MODULE_LOADING", "LAZY")

ROOT_DIR = os.path.abspath("../")
for p in (ROOT_DIR, os.path.join(ROOT_DIR, "archived_models")):
    if p not in sys.path:
        sys.path.append(p)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import tensorrt as trt
import transformers

from transformers import WhisperProcessor, WhisperForConditionalGeneration, WhisperConfig

DEVICE = "cuda"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| tensorrt", trt.__version__)
print("GPU:", torch.cuda.get_device_name(0))

torch 2.0.1+cu117 | transformers 4.24.0 | tensorrt 8.6.1
GPU: NVIDIA A100-SXM4-80GB


## 1. HuggingFace Whisper 모델 로드

In [2]:
# choices: openai/whisper-tiny | base | small | medium | large-v2
Whisper_VARIANT = os.environ.get("WHISPER_VARIANT", "openai/whisper-large-v2")

processor = WhisperProcessor.from_pretrained(Whisper_VARIANT)
tokenizer = processor.tokenizer
whisper_model = WhisperForConditionalGeneration.from_pretrained(Whisper_VARIANT).eval()
config = whisper_model.config

pytorch_model_dir = f"./models/{Whisper_VARIANT}/pytorch"
os.makedirs(pytorch_model_dir, exist_ok=True)
whisper_model.save_pretrained(pytorch_model_dir)
print("Pytorch Model saved to", pytorch_model_dir)

Pytorch Model saved to ./models/openai/whisper-large-v2/pytorch


## 2. 입력 오디오

- 영어: librispeech dummy 샘플
- 한국어: `korean_news.mp4` (PyAV로 16kHz mono 디코딩)

In [3]:
import io, itertools
import av

def decode_audio(input_file, sampling_rate=16000):
    """PyAV로 오디오/비디오 파일을 16kHz mono float32로 디코딩."""
    resampler = av.audio.resampler.AudioResampler(format="s16", layout="mono", rate=sampling_rate)
    raw_buffer = io.BytesIO()
    dtype = None
    with av.open(input_file, metadata_errors="ignore") as container:
        frames = container.decode(audio=0)
        frames = _ignore_invalid_frames(frames)
        frames = _group_frames(frames, 500000)
        frames = _resample_frames(frames, resampler)
        for frame in frames:
            array = frame.to_ndarray()
            dtype = array.dtype
            raw_buffer.write(array)
    audio = np.frombuffer(raw_buffer.getbuffer(), dtype=dtype)
    return audio.astype(np.float32) / 32768.0

def _ignore_invalid_frames(frames):
    iterator = iter(frames)
    while True:
        try:
            yield next(iterator)
        except StopIteration:
            break
        except av.error.InvalidDataError:
            continue

def _group_frames(frames, num_samples=None):
    fifo = av.audio.fifo.AudioFifo()
    for frame in frames:
        frame.pts = None  # Ignore timestamp check.
        fifo.write(frame)
        if num_samples is not None and fifo.samples >= num_samples:
            yield fifo.read()
    if fifo.samples > 0:
        yield fifo.read()

def _resample_frames(frames, resampler):
    # Add None to flush the resampler.
    for frame in itertools.chain(frames, [None]):
        yield from resampler.resample(frame)

In [4]:
from datasets import load_dataset

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
input_features = processor(ds[0]["audio"]["array"], sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)

ko_audio = decode_audio("korean_news.mp4")
print(f"korean_news.mp4: {ko_audio.shape[0] / 16000:.1f}s")
input_features_ko = processor(ko_audio, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)

forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe", no_timestamps=True)
forced_decoder_ids_ko = processor.get_decoder_prompt_ids(language="ko", task="transcribe", no_timestamps=True)
input_features.shape, input_features_ko.shape

korean_news.mp4: 81.2s


(torch.Size([1, 80, 3000]), torch.Size([1, 80, 3000]))

## 3. PyTorch(HF) 기준 결과

In [5]:
whisper_model.to(DEVICE).float()
max_output_len = config.max_length
min_output_len = 0
num_beams = 1

def hf_generate(features, forced_ids, use_cache=True):
    return whisper_model.generate(
        features, forced_decoder_ids=forced_ids, max_length=max_output_len,
        min_length=min_output_len, num_beams=num_beams, use_cache=use_cache,
    )

with torch.no_grad():
    outputs_hf = tokenizer.decode(hf_generate(input_features, forced_decoder_ids)[0], skip_special_tokens=True)
    outputs_hf_ko = tokenizer.decode(hf_generate(input_features_ko, forced_decoder_ids_ko)[0], skip_special_tokens=True)
print("EN:", outputs_hf)
print("KO:", outputs_hf_ko)

EN:  Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel.
KO:  제6호 태풍 카누는 여전히 매우 강한 세력을 유지한 채 북서지나고 있습니다. 하지만 이동 속도가 점점 느려져 거의 정체하는 모습입니다. 태풍은 동중국해에 머물다 동쪽으로 방향을 급격히 틀어 이동할 걸로 보입니다. 속도도 조금씩 빨라지며 다음 주 초중반에는 일본 규슈 남쪽 해상까지 진출하겠습니다. 세력도 크게 약화하지는 않을 전망입니다.


### HF generate 벤치마크

In [6]:
from NNDF.networks import TimingProfile, NetworkMetadata, Precision
from NNDF.general_utils import measure_python_inference_code

timing_profile = TimingProfile(iterations=10, number=1, warmup=1, duration=0, percentile=[50, 99])

def percentile_print(timing):
    return ", ".join(f"p{timing_profile.percentile[i]} {p * 1000:.2f}ms" for i, p in enumerate(timing))

with torch.no_grad():
    whisper_model.float()
    hf_nonkv_time = measure_python_inference_code(lambda: hf_generate(input_features, forced_decoder_ids, use_cache=False), timing_profile)
    hf_kv_time = measure_python_inference_code(lambda: hf_generate(input_features, forced_decoder_ids, use_cache=True), timing_profile)

    whisper_model.half()
    hf_nonkv_time_fp16 = measure_python_inference_code(lambda: hf_generate(input_features.half(), forced_decoder_ids, use_cache=False), timing_profile)
    hf_kv_time_fp16 = measure_python_inference_code(lambda: hf_generate(input_features.half(), forced_decoder_ids, use_cache=True), timing_profile)
    # 한국어 뉴스(30초 구간, 출력 토큰이 훨씬 많음)
    hf_kv_time_fp16_ko = measure_python_inference_code(lambda: hf_generate(input_features_ko.half(), forced_decoder_ids_ko, use_cache=True), timing_profile)
    whisper_model.float()

print("HF FP32 no-cache:", percentile_print(hf_nonkv_time), "| cache:", percentile_print(hf_kv_time))
print("HF FP16 no-cache:", percentile_print(hf_nonkv_time_fp16), "| cache:", percentile_print(hf_kv_time_fp16))
print("HF FP16 cache (KO):", percentile_print(hf_kv_time_fp16_ko))

HF FP32 no-cache: p50 1899.00ms, p99 2004.11ms | cache: p50 886.33ms, p99 931.16ms
HF FP16 no-cache: p50 965.03ms, p99 996.38ms | cache: p50 687.25ms, p99 733.19ms
HF FP16 cache (KO): p50 3134.51ms, p99 3787.04ms


### NNDF PyTorch 래퍼(인코더/디코더 분리) + greedy search

In [7]:
from Whisper.measurements import (
    decoder_inference as w_decoder_inference,
    encoder_inference as w_encoder_inference,
    full_inference_greedy,
)
from Whisper.export import WhisperEncoderTorchFile, WhisperDecoderTorchFile, WhisperEncoderTRTEngine, WhisperDecoderTRTEngine
from Whisper.WhisperModelConfig import WhisperModelTRTConfig, WhisperMetadata

HF_KV = True
decoder_input_ids = torch.full((1, 1), config.decoder_start_token_id, dtype=torch.int32, device=DEVICE)

def run_torch(precision_half):
    whisper_model.half() if precision_half else whisper_model.float()
    feats = input_features.half() if precision_half else input_features.float()
    enc = WhisperEncoderTorchFile.TorchModule(whisper_model.get_encoder())
    dec = WhisperDecoderTorchFile.TorchModule(whisper_model.get_decoder(), whisper_model.proj_out, config)
    with torch.no_grad():
        enc_out, enc_time = w_encoder_inference(enc, feats, timing_profile)
        _, dec_time = w_decoder_inference(dec, decoder_input_ids, enc_out, timing_profile, use_cache=HF_KV)
        ids, full_time = full_inference_greedy(
            enc, dec, feats, tokenizer, timing_profile,
            max_length=max_output_len, min_length=min_output_len,
            use_cache=HF_KV, forced_decoder_ids=forced_decoder_ids,
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True), enc_time, dec_time, full_time

outputs_pytorch, encoder_pytorch_time, decoder_pytorch_time, full_pytorch_time = run_torch(False)
outputs_pytorch_fp16, encoder_pytorch_time_fp16, decoder_pytorch_time_fp16, full_pytorch_time_fp16 = run_torch(True)
whisper_model.float()

print(f"PyTorch FP32 == HF? {outputs_pytorch == outputs_hf}")
print(f"PyTorch FP16 == HF? {outputs_pytorch_fp16 == outputs_hf}")
print(f"FP32 enc {percentile_print(encoder_pytorch_time)} | dec {percentile_print(decoder_pytorch_time)} | e2e {percentile_print(full_pytorch_time)}")
print(f"FP16 enc {percentile_print(encoder_pytorch_time_fp16)} | dec {percentile_print(decoder_pytorch_time_fp16)} | e2e {percentile_print(full_pytorch_time_fp16)}")

PyTorch FP32 == HF? True
PyTorch FP16 == HF? True
FP32 enc p50 276.35ms, p99 336.70ms | dec p50 46.65ms, p99 70.76ms | e2e p50 894.56ms, p99 931.05ms
FP16 enc p50 98.91ms, p99 154.40ms | dec p50 24.12ms, p99 24.37ms | e2e p50 710.84ms, p99 765.76ms


## 4. ONNX 변환 (opset 17, KV cache 구조)

- `encoder`: `input_features (1, 80, 3000)` → `cross_k, cross_v (L, 1, H, 1500, D)`
- `decoder`: `input_ids (1, T), self_k, self_v (L, 1, H, P, D), cross_k, cross_v` → `logits (1, V)`, `present_k, present_v (L, 1, H, P+T, D)`

모델이 2GB를 넘어 external data로 저장되므로 encoder/decoder를 서로 다른 폴더에 export한다.

In [8]:
from Whisper import trt_kv

kv_onnx_dir = f"./models/{Whisper_VARIANT}/onnx-kv-opset{trt_kv.ONNX_OPSET}"
kv_encoder_onnx = os.path.join(kv_onnx_dir, "encoder", "encoder-crosskv.onnx")
kv_decoder_onnx = os.path.join(kv_onnx_dir, "decoder", "decoder-kv.onnx")

force_onnx = False
if force_onnx or not (os.path.exists(kv_encoder_onnx) and os.path.exists(kv_decoder_onnx)):
    trt_kv.export_onnx(whisper_model, kv_encoder_onnx, kv_decoder_onnx)  # CPU / FP32에서 export

# TRT 단계에서 GPU 메모리를 쓰도록 HF 모델은 CPU로 내린다
whisper_model.float().cpu()
torch.cuda.empty_cache()

import onnx
for path in (kv_encoder_onnx, kv_decoder_onnx):
    ops = {n.op_type for n in onnx.load(path, load_external_data=False).graph.node}
    print(os.path.basename(path), "| LayerNormalization:", "LayerNormalization" in ops, "| Pow:", "Pow" in ops)

============= Diagnostic Run torch.onnx.export version 2.0.1+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



============= Diagnostic Run torch.onnx.export version 2.0.1+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

encoder-crosskv.onnx | LayerNormalization: True | Pow: False
decoder-kv.onnx | LayerNormalization: True | Pow: False


## 5. TensorRT 엔진 빌드 (FP32 / FP16)

FP16 엔진은 KV cache 입출력도 FP16으로 두어 매 step의 FP32↔FP16 변환 복사를 없앤다. 엔진 파일이 있으면 재사용 (`force_build=True`로 재빌드).

In [9]:
import time

kv_engine_dir = f"./models/{Whisper_VARIANT}/tensorrt-kv-opset{trt_kv.ONNX_OPSET}"
force_build = False

kv_engines = {}
for name, fp16 in [("FP32", False), ("FP16", True)]:
    t0 = time.time()
    kv_engines[name] = trt_kv.build_whisper_engines(
        config, kv_encoder_onnx, kv_decoder_onnx, kv_engine_dir, fp16=fp16, force=force_build
    )
    print(f"{name}: {time.time() - t0:.0f}s", *kv_engines[name], sep="\n  ")

[10/19/2026-12:04:10] [TRT] [W] onnx2trt_utils.cpp:374: Your ONNX model has been generated with INT64 weights, while TensorRT does not natively support INT64. Attempting to cast down to INT32.


[10/19/2026-12:07:36] [TRT] [W] onnx2trt_utils.cpp:374: Your ONNX model has been generated with INT64 weights, while TensorRT does not natively support INT64. Attempting to cast down to INT32.


FP32: 483s
  ./models/openai/whisper-large-v2/tensorrt-kv-opset17/encoder-crosskv-fp32-bs1.engine
  ./models/openai/whisper-large-v2/tensorrt-kv-opset17/decoder-kv-fp32-bs1.engine


[10/19/2026-12:12:15] [TRT] [W] onnx2trt_utils.cpp:374: Your ONNX model has been generated with INT64 weights, while TensorRT does not natively support INT64. Attempting to cast down to INT32.


[10/19/2026-12:19:28] [TRT] [W] TensorRT encountered issues when converting weights between types and that could affect accuracy.
[10/19/2026-12:19:28] [TRT] [W] If this is not the desired behavior, please modify the weights or retrain with regularization to adjust the magnitude of the weights.
[10/19/2026-12:19:28] [TRT] [W] Check verbose logs for the list of affected weights.
[10/19/2026-12:19:28] [TRT] [W] - 490 weights are affected by this issue: Detected subnormal FP16 values.


[10/19/2026-12:19:44] [TRT] [W] onnx2trt_utils.cpp:374: Your ONNX model has been generated with INT64 weights, while TensorRT does not natively support INT64. Attempting to cast down to INT32.


[10/19/2026-12:31:27] [TRT] [W] TensorRT encountered issues when converting weights between types and that could affect accuracy.
[10/19/2026-12:31:27] [TRT] [W] If this is not the desired behavior, please modify the weights or retrain with regularization to adjust the magnitude of the weights.
[10/19/2026-12:31:27] [TRT] [W] Check verbose logs for the list of affected weights.
[10/19/2026-12:31:27] [TRT] [W] - 534 weights are affected by this issue: Detected subnormal FP16 values.


FP16: 1168s
  ./models/openai/whisper-large-v2/tensorrt-kv-opset17/encoder-crosskv-fp16-bs1.engine
  ./models/openai/whisper-large-v2/tensorrt-kv-opset17/decoder-kv-fp16-bs1.engine


## 6. TensorRT 추론 (KV cache)

In [10]:
# 인코더 + cross K/V 정확도: PyTorch FP32 기준 (FP16 overflow가 있으면 inf/NaN이 나온다)
ref_model = whisper_model  # CPU / FP32
with torch.no_grad():
    ref_k, ref_v = trt_kv.WhisperEncoderCrossKV(ref_model)(input_features_ko.cpu())

for name, (enc_path, dec_path) in kv_engines.items():
    enc = trt_kv.TRTModule(enc_path)
    k, v = [t.float().cpu() for t in enc(input_features=input_features_ko)]
    print(f"TRT {name} cross_k max|diff| {(k - ref_k).abs().max():.4f} | cross_v max|diff| {(v - ref_v).abs().max():.4f}"
          f" | finite: {bool(torch.isfinite(k).all() and torch.isfinite(v).all())}")
    del enc
del ref_k, ref_v

TRT FP32 cross_k max|diff| 0.3821 | cross_v max|diff| 0.3290 | finite: True


TRT FP16 cross_k max|diff| 8.3610 | cross_v max|diff| 5.1562 | finite: True


In [11]:
prompt_en = [config.decoder_start_token_id] + [t for _, t in forced_decoder_ids]
prompt_ko = [config.decoder_start_token_id] + [t for _, t in forced_decoder_ids_ko]

def sync(fn):
    def _f():
        out = fn()
        torch.cuda.synchronize()
        return out
    return _f

kv_results = {}
for name, (enc_path, dec_path) in kv_engines.items():
    runner = trt_kv.WhisperTRTKV(enc_path, dec_path, config)

    toks_en = runner.generate(input_features, prompt_en)
    toks_ko = runner.generate(input_features_ko, prompt_ko)
    text_en = tokenizer.decode(toks_en, skip_special_tokens=True)
    text_ko = tokenizer.decode(toks_ko, skip_special_tokens=True)

    enc_time = measure_python_inference_code(sync(lambda: runner.encode(input_features)), timing_profile)
    e2e_en = measure_python_inference_code(sync(lambda: runner.generate(input_features, prompt_en)), timing_profile)
    e2e_ko = measure_python_inference_code(sync(lambda: runner.generate(input_features_ko, prompt_ko)), timing_profile)
    n_en, n_ko = len(toks_en) - len(prompt_en), len(toks_ko) - len(prompt_ko)

    kv_results[name] = dict(
        text_en=text_en, text_ko=text_ko, enc=enc_time, e2e_en=e2e_en, e2e_ko=e2e_ko,
        tok_ms=(e2e_ko[0] - enc_time[0]) / n_ko * 1000, n_en=n_en, n_ko=n_ko,
    )
    print(f"[TRT-KV {name}] EN == HF? {text_en == outputs_hf} | {text_en}")
    print(f"[TRT-KV {name}] KO == HF? {text_ko == outputs_hf_ko} | {text_ko}")
    print(f"[TRT-KV {name}] encoder {percentile_print(enc_time)} | E2E EN({n_en} tok) {percentile_print(e2e_en)}"
          f" | E2E KO({n_ko} tok) {percentile_print(e2e_ko)} | decoder {kv_results[name]['tok_ms']:.2f} ms/token\n")
    del runner
    torch.cuda.empty_cache()

[TRT-KV FP32] EN == HF? True |  Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel.
[TRT-KV FP32] KO == HF? True |  제6호 태풍 카누는 여전히 매우 강한 세력을 유지한 채 북서지나고 있습니다. 하지만 이동 속도가 점점 느려져 거의 정체하는 모습입니다. 태풍은 동중국해에 머물다 동쪽으로 방향을 급격히 틀어 이동할 걸로 보입니다. 속도도 조금씩 빨라지며 다음 주 초중반에는 일본 규슈 남쪽 해상까지 진출하겠습니다. 세력도 크게 약화하지는 않을 전망입니다.
[TRT-KV FP32] encoder p50 45.68ms, p99 46.15ms | E2E EN(22 tok) p50 272.24ms, p99 289.40ms | E2E KO(126 tok) p50 1894.31ms, p99 3145.30ms | decoder 14.67 ms/token



[TRT-KV FP16] EN == HF? True |  Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel.
[TRT-KV FP16] KO == HF? True |  제6호 태풍 카누는 여전히 매우 강한 세력을 유지한 채 북서지나고 있습니다. 하지만 이동 속도가 점점 느려져 거의 정체하는 모습입니다. 태풍은 동중국해에 머물다 동쪽으로 방향을 급격히 틀어 이동할 걸로 보입니다. 속도도 조금씩 빨라지며 다음 주 초중반에는 일본 규슈 남쪽 해상까지 진출하겠습니다. 세력도 크게 약화하지는 않을 전망입니다.
[TRT-KV FP16] encoder p50 20.15ms, p99 31.63ms | E2E EN(22 tok) p50 314.91ms, p99 338.10ms | E2E KO(126 tok) p50 1582.60ms, p99 1636.52ms | decoder 12.40 ms/token



## 7. 결과 요약

`TensorRT (no KV, opset 12)` 행은 v1(NNDF non-KV 엔진)에서 GPU가 비어 있을 때 측정한 값이다.

⚠️ 2026-10-19 실행은 8개 GPU 모두 다른 학습이 60~70% 사용 중인 상태에서 측정 → HF 기준선도 v1보다 1.3~1.7배 느리게 나옴. 같은 조건 microbenchmark에서 디코더 1 step은 KV(FP16) min 8.6ms / non-KV(FP16) min 7.6ms로 둘 다 GPU 공유 영향이 커서 속도 비교는 빈 GPU에서 다시 측정해야 함.

In [12]:
from tabulate import tabulate

ms = lambda t: f"{t[0] * 1000:.2f}"
r32, r16 = kv_results["FP32"], kv_results["FP16"]
data = [
    ["Framework", "Precision", "Encoder p50 (ms)", "E2E EN p50 (ms)", "E2E KO p50 (ms)", "EN == HF", "KO == HF"],
    ["HuggingFace (w/o cache)", "FP32", "-", ms(hf_nonkv_time), "-", "-", "-"],
    ["HuggingFace (w/ cache)", "FP32", "-", ms(hf_kv_time), "-", "-", "-"],
    ["HuggingFace (w/ cache)", "FP16", "-", ms(hf_kv_time_fp16), ms(hf_kv_time_fp16_ko), "-", "-"],
    ["PyTorch (NNDF wrapper)", "FP16", ms(encoder_pytorch_time_fp16), ms(full_pytorch_time_fp16), "-", outputs_pytorch_fp16 == outputs_hf, "-"],
    ["TensorRT (no KV, opset 12)", "FP16", "41.99", "242.83", "-", True, True],
    ["TensorRT KV (opset 17)", "FP32", ms(r32["enc"]), ms(r32["e2e_en"]), ms(r32["e2e_ko"]), r32["text_en"] == outputs_hf, r32["text_ko"] == outputs_hf_ko],
    ["TensorRT KV (opset 17)", "FP16", ms(r16["enc"]), ms(r16["e2e_en"]), ms(r16["e2e_ko"]), r16["text_en"] == outputs_hf, r16["text_ko"] == outputs_hf_ko],
]
print(f"Device: {torch.cuda.get_device_name()} | variant: {Whisper_VARIANT} | 출력 토큰 EN {r16['n_en']}, KO {r16['n_ko']}")
print(tabulate(data, headers="firstrow", tablefmt="github"))
print()
print(f"speedup vs HF FP16 cache — EN {hf_kv_time_fp16[0] / r16['e2e_en'][0]:.1f}x, KO {hf_kv_time_fp16_ko[0] / r16['e2e_ko'][0]:.1f}x")
print(f"decoder ms/token — TRT-KV FP32 {r32['tok_ms']:.2f}, FP16 {r16['tok_ms']:.2f}")

Device: NVIDIA A100-SXM4-80GB | variant: openai/whisper-large-v2 | 출력 토큰 EN 22, KO 126
| Framework                  | Precision   | Encoder p50 (ms)   |   E2E EN p50 (ms) | E2E KO p50 (ms)   | EN == HF   | KO == HF   |
|----------------------------|-------------|--------------------|-------------------|-------------------|------------|------------|
| HuggingFace (w/o cache)    | FP32        | -                  |           1899    | -                 | -          | -          |
| HuggingFace (w/ cache)     | FP32        | -                  |            886.33 | -                 | -          | -          |
| HuggingFace (w/ cache)     | FP16        | -                  |            687.25 | 3134.51           | -          | -          |
| PyTorch (NNDF wrapper)     | FP16        | 98.91              |            710.84 | -                 | True       | -          |
| TensorRT (no KV, opset 12) | FP16        | 41.99              |            242.83 | -                 | True       | Tr